# Profiling the FM solver


In [1]:
import time
import torch

from nami.distributions.normal import StandardNormal
from nami.divergence.hutchinson import HutchinsonDivergence
from nami.fields.base import VectorField
from nami.fields.velocity import VelocityField
from nami.processes.fm import FlowMatching
from nami.solvers.heun import Heun

torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") # we need a gpu for this study later on, but setup here
print("device:", DEVICE, "| torch:", torch.__version__)

device: cpu | torch: 2.10.0


In [2]:
def timeit(fn, *, warmup=3, iters=10):
    """Mean ms/iter, with CUDA sync if relevant."""
    
    for _ in range(warmup):
        fn() # this is the function we want to time
        
    if DEVICE.type == "cuda":
        # syncronisation point for the gpu
        torch.cuda.synchronize()
        
    t0 = time.perf_counter()
    
    for _ in range(iters):
        fn()
        
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    
    # calculate the average time per iteration
    return (time.perf_counter() - t0) / iters * 1e3


def build(dim=8, 
          hidden=256, 
          layers=3, 
          steps=32, 
          field=None):
    
    if field is None:
        field = VelocityField(dim, hidden=hidden, layers=layers)
        
    field = field.to(DEVICE).eval()
    
    base = StandardNormal(dim, device=DEVICE)
    
    return FlowMatching(field, base, Heun(steps=steps), validate_args=False)(None), field

### Baseline

In [3]:
DIM, BATCH, STEPS = 8, 1024, 32


est = HutchinsonDivergence()


# build the process and field
proc, field = build(dim=DIM, steps=STEPS)

# sample and check the log_prob
x = proc.sample((BATCH,))
print("x:", tuple(x.shape), "| log_prob finite:",
      # we need isfinite to check that the log_prob is finite, otherwise inf is a useless returned value
      bool(torch.isfinite(proc.log_prob(x, estimator=est)).all()))

t_sample = timeit(lambda: proc.sample((BATCH,)))
t_logp = timeit(lambda: proc.log_prob(x, estimator=est))

print(f"sample   : {t_sample:8.2f} ms/iter  ({STEPS} steps, {STEPS} NFE)")
print(f"log_prob : {t_logp:8.2f} ms/iter  ({STEPS} steps, {2*STEPS} NFE + vJp)")

x: (1024, 8) | log_prob finite: True
sample   :   158.82 ms/iter  (32 steps, 32 NFE)
log_prob :   532.52 ms/iter  (32 steps, 64 NFE + vJp)


### Time spent

In [4]:
class NullField(VectorField):
    def __init__(self, dim):
        super().__init__()
        self.event_shape = (dim,)
        
        # one tiny param so .to(device) / autograd graph still works
        self._p = torch.nn.Parameter(torch.zeros(1))

    @property
    def event_ndim(self):
        return 1

    def forward(self, x, t, c=None):
        return x * self._p  # zero, but differentiable wrt x for the vJp path


null_proc, _ = build(dim=DIM, steps=STEPS, field=NullField(DIM))
t_sample_null = timeit(lambda: null_proc.sample((BATCH,)))
t_logp_null = timeit(lambda: null_proc.log_prob(x, estimator=est))

print(f"{'':16}{'real':>12}{'null (loop only)':>20}{'network/autograd':>20}")
print(f"{'sample':16}{t_sample:12.2f}{t_sample_null:20.2f}{t_sample - t_sample_null:20.2f}")
print(f"{'log_prob':16}{t_logp:12.2f}{t_logp_null:20.2f}{t_logp - t_logp_null:20.2f}")

                        real    null (loop only)    network/autograd
sample                158.82                1.58              157.24
log_prob              532.52                5.95              526.57


### just `torch.compile`

Compiling the field fuses elementwise ops and cuts kernel-launch overhead.

In [5]:
try:
    c_proc, _ = build(dim=DIM, steps=STEPS,
                      field=torch.compile(VelocityField(DIM, hidden=256, layers=3))
                    )
    
    
    # warmup triggers compilation
    _ = c_proc.sample((BATCH,))
    
    _ = c_proc.log_prob(x, estimator=est)
    
    t_sample_c = timeit(lambda: c_proc.sample((BATCH,)))
    
    t_logp_c = timeit(lambda: c_proc.log_prob(x, estimator=est))
    
    print(f"sample   : {t_sample:7.2f} -> {t_sample_c:7.2f} ms  ({t_sample/t_sample_c:.2f}x)")
    print(f"log_prob : {t_logp:7.2f} -> {t_logp_c:7.2f} ms  ({t_logp/t_logp_c:.2f}x)")
    
    
except Exception as e:
    print("torch.compile unavailable in this env:", type(e).__name__, e)

sample   :  158.82 ->  176.53 ms  (0.90x)
log_prob :  532.52 ->  802.48 ms  (0.66x)


### `torch.profiler` op breakdown

Confirms the dominant ops are matmul / addmm (the MLP) and the autograd backward

In [10]:
from torch.profiler import ProfilerActivity, profile

acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if DEVICE.type == "cuda" else [])

with profile(activities=acts, # record both CPU and GPU
             record_shapes=False, # don't record the shapes of the operations
             acc_events=True, # accumulate events
             with_stack=True) as prof: # record the stack trace of the operations
    
    proc.log_prob(x, estimator=est)
    
sort_key = "cuda_time_total" if DEVICE.type == "cuda" else "cpu_time_total"

print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::linear         0.16%     624.884us        46.92%     188.029ms     367.244us           512  
                                            aten::addmm        35.97%     144.117ms        46.36%     185.779ms     362.849us           512  
    autograd::engine::evaluate_function: AddmmBackward0         1.45%       5.816ms        22.71%      90.991ms     355.433us           256  
                                         AddmmBackward0         0.26%       1.030ms        21.26%      85.175ms     332.714us           256  
      